There is no single command to get the table name and the size next to it, but every catalogue has information schema and that has again the tables and columns of each table under that particular scheme, so we can look through it and get the size of each table

In [0]:
catalog = 'samples'
schema = 'tpcds_sf1'

In [0]:
tables_df = spark.sql(f'''
                      select table_name from {catalog}.information_schema.tables
                      where table_schema = '{schema}' and table_type != 'VIEW'
                      ''')
display(tables_df)

In [0]:
table_sizes = []
# get the name name from the ROW object
for row in tables_df.collect():
    table_name = row["table_name"]
    detail = spark.sql(f"DESCRIBE DETAIL {catalog}.{schema}.{table_name}").collect()[0]
    
    # Extract size in bytes and convert to Megabytes (MB).
    size_in_mb = detail["sizeInBytes"] / (1024 * 1024)
    table_sizes.append((table_name, round(size_in_mb, 2)))

In [0]:
# 4. Create and display a DataFrame (a structured, distributed table) with the results.
result_df = spark.createDataFrame(table_sizes, ["Table Name", "Size (MB)"])
display(result_df)